## Exercise 3: OLTP vs. OLAP Use Cases: Practice with Postgres and DuckDB

In this exercise, we will work to better understand and demonstrate the performance differences between a row-oriented database (PostgreSQL) optimized for OLTP workloads and a column-oriented database (DuckDB) optimized for OLAP workloads. Note that we've already seen some of this in the past, but now we will dive into more detail, and test this system for the OLAP use case as well!

## Visualizing Data Storage
Examine the sample sales data table in this notebook below, which we will use to illustrate the fundamental difference between how row-based and column-based databases store data on disk.

| OrderID | Product  | Quantity | Price | SaleDate   |
|---------|----------|----------|-------|------------|
| 101     | Laptop   | 1        | 1200  | 2025-10-14 |
| 102     | Mouse    | 2        | 25    | 2025-10-14 |
| 103     | Keyboard | 1        | 75    | 2025-10-15 |

 
A) Row-Oriented Storage (PostgreSQL) 💾
In systems such as PostgreSQL, data for each record is stored together, making it fast to retrieve an entire row. This also makes it substantially faster to write an entire row, given that all the relevant required information is present and provided by the application using the database.

- [101, 'Laptop', 1, 1200, '2025-10-14'] 

- [102, 'Mouse', 2, 25, '2025-10-14'] 

- [103, 'Keyboard', 1, 75, '2025-10-15']

B) Column-Oriented Storage (DuckDB) 📊
Data is stored by column, making it highly efficient to read and aggregate data from a few specific columns.

- [101, 102, 103] 

- ['Laptop', 'Mouse', 'Keyboard']

- [1, 2, 1] 

- [1200, 25, 75]

- ['2025-10-14', '2025-10-14', '2025-10-15']

We often will see row oriented systems called "OLTP" databases and column oriented systems, called "OLAP" database or data warehouses. 

Question: If you wanted to calculate the total revenue (sum of Quantity * Price), which storage format would likely be faster and why? The answer would typically be the column oriented approach, which allows you to ignore the irrelevant fields. Although if you wanted to write data, the row oriented approach would be faster, as you wouldn't have to write to each separate column component.

let's use our Python notebook to connect to PostgreSQL and DuckDB, run benchmark tests, and see the performance difference firsthand. The industry actually has data which can be used for this benchmarking purpose specifically. These datasets are called TPC-C and TPC-H.

- TPC-C: an On-Line Transaction Processing Benchmark - https://www.tpc.org/tpcc/
- TPC-H: a Decision Support Benchmark (OLAP) - https://www.tpc.org/tpch/

The OLTP TPC-C tests are actually quite complex, and involve the creation of multiple simulated users, multiple writes per user, and then the deletion of those users. This can often be done using software such as HammerDB (an industry open source benchmarking tool used to test systems like cloud hosted databases, such as AWS RDS). For now, we've provided scripts that simulate the TPC-C results. You can view their output by running the cells below, and examine the benchmark. 

In [1]:
%pip install Faker
import psycopg2
import duckdb
import pandas as pd
import random
import time
from faker import Faker
from datetime import datetime, timedelta
import sys
import io

Note: you may need to restart the kernel to use updated packages.


In [2]:
# --- Configuration ---
# Scale factor (1 = 1 warehouse)
# WARNING: Increasing this scales data exponentially. 
# 2 is a good test. 5+ will be very large.
SCALE_FACTOR = 2

# Constants based on TPC-C Spec
DISTRICTS_PER_WAREHOUSE = 10
CUSTOMERS_PER_DISTRICT = 3000
ITEMS = 100000
# ---------------------

fake = Faker()

def create_pg_connection():
    """Creates a psycopg2 connection to PostgreSQL and ensures 'tpcc' schema exists."""
    try:
        conn = psycopg2.connect(
            dbname='postgres',
            user='postgres',
            password='I4NAennDnRh1uO9z2y0k6pMfAf',
            host='localhost',
            port='5432'
        )
        conn.autocommit = False # We want to control transactions
        
        with conn.cursor() as cursor:
            cursor.execute("CREATE SCHEMA IF NOT EXISTS tpcc;")
        conn.commit()
        print("Successfully connected to PostgreSQL and ensured 'tpcc' schema exists.")
        return conn
    except Exception as e:
        print(f"Error: Could not connect to PostgreSQL: {e}")
        sys.exit(1)

def connect_to_duckdb():
    """Connects to the DuckDB database file."""
    try:
        conn = duckdb.connect(database='tpcc.duckdb')
        print("Successfully connected to DuckDB at 'tpcc.duckdb'.")
        return conn
    except Exception as e:
        print(f"Error: Could not connect to DuckDB: {e}")
        sys.exit(1)

def create_tpcc_schema(conn, is_postgres=True):
    """
    Creates the 9 TPC-C tables in the target database.
    """
    schema_prefix = "tpcc." if is_postgres else ""
    
    # Use 'TEXT' for simplicity; PG/DuckDB handle it well.
    # Use 'DOUBLE PRECISION' for 'REAL' in DuckDB.
    real_type = "DECIMAL(10, 4)" if is_postgres else "DOUBLE"
    num_type = "DECIMAL(12, 2)" if is_postgres else "DOUBLE"
    
    # Primary/Foreign keys are omitted for faster loading and simplicity.
    # In a real scenario, these are critical.
    
    queries = [
        f"""CREATE TABLE IF NOT EXISTS {schema_prefix}warehouse (
            w_id INT NOT NULL,
            w_name TEXT,
            w_street_1 TEXT,
            w_street_2 TEXT,
            w_city TEXT,
            w_state CHAR(2),
            w_zip CHAR(9),
            w_tax {real_type},
            w_ytd {num_type}
        );""",
        
        f"""CREATE TABLE IF NOT EXISTS {schema_prefix}district (
            d_id INT NOT NULL,
            d_w_id INT NOT NULL,
            d_name TEXT,
            d_street_1 TEXT,
            d_street_2 TEXT,
            d_city TEXT,
            d_state CHAR(2),
            d_zip CHAR(9),
            d_tax {real_type},
            d_ytd {num_type},
            d_next_o_id INT
        );""",

        f"""CREATE TABLE IF NOT EXISTS {schema_prefix}customer (
            c_id INT NOT NULL,
            c_d_id INT NOT NULL,
            c_w_id INT NOT NULL,
            c_first TEXT,
            c_middle CHAR(2),
            c_last TEXT,
            c_street_1 TEXT,
            c_street_2 TEXT,
            c_city TEXT,
            c_state CHAR(2),
            c_zip CHAR(9),
            c_phone CHAR(16),
            c_since TIMESTAMP,
            c_credit CHAR(2),
            c_credit_lim {num_type},
            c_discount {real_type},
            c_balance {num_type},
            c_ytd_payment {num_type},
            c_payment_cnt INT,
            c_delivery_cnt INT,
            c_data TEXT
        );""",
        
        f"""CREATE TABLE IF NOT EXISTS {schema_prefix}history (
            h_c_id INT,
            h_c_d_id INT,
            h_c_w_id INT,
            h_d_id INT,
            h_w_id INT,
            h_date TIMESTAMP,
            h_amount {num_type},
            h_data TEXT
        );""",

        f"""CREATE TABLE IF NOT EXISTS {schema_prefix}item (
            i_id INT NOT NULL,
            i_im_id INT,
            i_name TEXT,
            i_price {num_type},
            i_data TEXT
        );""",
        
        f"""CREATE TABLE IF NOT EXISTS {schema_prefix}stock (
            s_i_id INT NOT NULL,
            s_w_id INT NOT NULL,
            s_quantity INT,
            s_dist_01 CHAR(24),
            s_dist_02 CHAR(24),
            s_dist_03 CHAR(24),
            s_dist_04 CHAR(24),
            s_dist_05 CHAR(24),
            s_dist_06 CHAR(24),
            s_dist_07 CHAR(24),
            s_dist_08 CHAR(24),
            s_dist_09 CHAR(24),
            s_dist_10 CHAR(24),
            s_ytd INT,
            s_order_cnt INT,
            s_remote_cnt INT,
            s_data TEXT
        );""",

        f"""CREATE TABLE IF NOT EXISTS {schema_prefix}orders (
            o_id INT NOT NULL,
            o_d_id INT NOT NULL,
            o_w_id INT NOT NULL,
            o_c_id INT,
            o_entry_d TIMESTAMP,
            o_carrier_id INT,
            o_ol_cnt INT,
            o_all_local INT
        );""",
        
        f"""CREATE TABLE IF NOT EXISTS {schema_prefix}new_order (
            no_o_id INT NOT NULL,
            no_d_id INT NOT NULL,
            no_w_id INT NOT NULL
        );""",
        
        f"""CREATE TABLE IF NOT EXISTS {schema_prefix}order_line (
            ol_o_id INT NOT NULL,
            ol_d_id INT NOT NULL,
            ol_w_id INT NOT NULL,
            ol_number INT NOT NULL,
            ol_i_id INT,
            ol_supply_w_id INT,
            ol_delivery_d TIMESTAMP,
            ol_quantity INT,
            ol_amount {num_type},
            ol_dist_info CHAR(24)
        );"""
    ]
    
    print(f"Creating 9 TPC-C tables ({'PostgreSQL' if is_postgres else 'DuckDB'})...")
    if is_postgres:
        with conn.cursor() as cursor:
            for q in queries:
                cursor.execute(q)
        conn.commit()
    else:
        for q in queries:
            conn.execute(q)
    print("Tables created.")

def generate_mock_data():
    """
    Generates mock data for TPC-C tables and returns dict of DataFrames.
    """
    print(f"Generating mock data for {SCALE_FACTOR} warehouse(s)...")
    
    # 1. Item
    items = []
    for i in range(1, ITEMS + 1):
        items.append({
            "i_id": i,
            "i_im_id": random.randint(1, 10000),
            "i_name": fake.text(max_nb_chars=24),
            "i_price": round(random.uniform(1.00, 100.00), 2),
            "i_data": fake.text(max_nb_chars=50)
        })
    df_item = pd.DataFrame(items)

    # 2. Warehouse
    warehouses = []
    for w in range(1, SCALE_FACTOR + 1):
        warehouses.append({
            "w_id": w,
            "w_name": fake.company(),
            "w_street_1": fake.street_address(),
            "w_street_2": fake.secondary_address(),
            "w_city": fake.city(),
            "w_state": fake.state_abbr(),
            "w_zip": fake.zipcode(),
            "w_tax": round(random.uniform(0.0000, 0.2000), 4),
            "w_ytd": 300000.00
        })
    df_warehouse = pd.DataFrame(warehouses)
    
    districts = []
    customers = []
    stocks = []
    
    for w_id in range(1, SCALE_FACTOR + 1):
        # 3. Stock
        for i_id in range(1, ITEMS + 1):
            stocks.append({
                "s_i_id": i_id,
                "s_w_id": w_id,
                "s_quantity": random.randint(10, 100),
                "s_dist_01": fake.password(length=24),
                "s_dist_02": fake.password(length=24),
                "s_dist_03": fake.password(length=24),
                "s_dist_04": fake.password(length=24),
                "s_dist_05": fake.password(length=24),
                "s_dist_06": fake.password(length=24),
                "s_dist_07": fake.password(length=24),
                "s_dist_08": fake.password(length=24),
                "s_dist_09": fake.password(length=24),
                "s_dist_10": fake.password(length=24),
                "s_ytd": 0,
                "s_order_cnt": 0,
                "s_remote_cnt": 0,
                "s_data": fake.text(max_nb_chars=50)
            })
            
        for d_id in range(1, DISTRICTS_PER_WAREHOUSE + 1):
            # 4. District
            districts.append({
                "d_id": d_id,
                "d_w_id": w_id,
                "d_name": fake.city(),
                "d_street_1": fake.street_address(),
                "d_street_2": fake.secondary_address(),
                "d_city": fake.city(),
                "d_state": fake.state_abbr(),
                "d_zip": fake.zipcode(),
                "d_tax": round(random.uniform(0.0000, 0.2000), 4),
                "d_ytd": 30000.00,
                "d_next_o_id": 3001 # All districts start at order 3001
            })
            
            # 5. Customer
            for c_id in range(1, CUSTOMERS_PER_DISTRICT + 1):
                customers.append({
                    "c_id": c_id,
                    "c_d_id": d_id,
                    "c_w_id": w_id,
                    "c_first": fake.first_name(),
                    "c_middle": "OE",
                    "c_last": fake.last_name(),
                    "c_street_1": fake.street_address(),
                    "c_street_2": fake.secondary_address(),
                    "c_city": fake.city(),
                    "c_state": fake.state_abbr(),
                    "c_zip": fake.zipcode(),
                    "c_phone": fake.phone_number(),
                    "c_since": fake.date_time_this_decade(),
                    "c_credit": "GC", # Good Credit
                    "c_credit_lim": 50000.00,
                    "c_discount": round(random.uniform(0.0000, 0.5000), 4),
                    "c_balance": -10.00,
                    "c_ytd_payment": 10.00,
                    "c_payment_cnt": 1,
                    "c_delivery_cnt": 0,
                    "c_data": fake.text(max_nb_chars=100) # Simplified
                })

    df_district = pd.DataFrame(districts)
    df_customer = pd.DataFrame(customers)
    df_stock = pd.DataFrame(stocks)
    
    # We will generate orders, order_line, new_order, history on the fly in the runner
    # But we create empty tables here.
    df_history = pd.DataFrame(columns=["h_c_id", "h_c_d_id", "h_c_w_id", "h_d_id", "h_w_id", "h_date", "h_amount", "h_data"])
    df_orders = pd.DataFrame(columns=["o_id", "o_d_id", "o_w_id", "o_c_id", "o_entry_d", "o_carrier_id", "o_ol_cnt", "o_all_local"])
    df_new_order = pd.DataFrame(columns=["no_o_id", "no_d_id", "no_w_id"])
    df_order_line = pd.DataFrame(columns=["ol_o_id", "ol_d_id", "ol_w_id", "ol_number", "ol_i_id", "ol_supply_w_id", "ol_delivery_d", "ol_quantity", "ol_amount", "ol_dist_info"])

    print("Mock data generated.")
    return {
        "warehouse": df_warehouse,
        "district": df_district,
        "customer": df_customer,
        "history": df_history,
        "item": df_item,
        "stock": df_stock,
        "orders": df_orders,
        "new_order": df_new_order,
        "order_line": df_order_line
    }

def load_data_postgres(conn, tables_df):
    """Loads data into PostgreSQL using psycopg2.copy_expert."""
    print("Loading data into PostgreSQL...")
    start_time = time.time()
    
    # Load order matters for real FKs, but not here.
    table_names = ["warehouse", "district", "customer", "history", "item", "stock", "orders", "new_order", "order_line"]
    
    with conn.cursor() as cursor:
        for table in table_names:
            df = tables_df[table]
            if df.empty:
                print(f"  Skipping empty table: {table}")
                continue
                
            print(f"  Loading {table} ({len(df)} rows)...")
            
            buffer = io.StringIO()
            df.to_csv(buffer, index=False, header=False, sep='\t', na_rep='\\N')
            buffer.seek(0)
            
            try:
                # Truncate before loading
                cursor.execute(f"TRUNCATE tpcc.{table} CASCADE;")
                copy_sql = f"COPY tpcc.{table} FROM STDIN WITH (FORMAT CSV, DELIMITER E'\\t', NULL '\\N')"
                cursor.copy_expert(copy_sql, buffer)
            except Exception as e:
                print(f"Error loading table {table}: {e}")
                conn.rollback()
                return
    
    conn.commit()
    end_time = time.time()
    print(f"PostgreSQL load complete in {end_time - start_time:.2f} seconds.")

def load_data_duckdb(conn, tables_df):
    """Loads data into DuckDB using DataFrames."""
    print("Loading data into DuckDB...")
    start_time = time.time()
    
    table_names = ["warehouse", "district", "customer", "history", "item", "stock", "orders", "new_order", "order_line"]
    
    try:
        for table in table_names:
            df = tables_df[table]
            if df.empty:
                print(f"  Skipping empty table: {table}")
                continue
                
            print(f"  Loading {table} ({len(df)} rows)...")
            # Register DataFrame as a temporary view
            conn.register(f'df_{table}', df)
            # Truncate and insert
            conn.execute(f"TRUNCATE {table};")
            conn.execute(f"INSERT INTO {table} SELECT * FROM df_{table};")
            # Unregister view
            conn.unregister(f'df_{table}')
            
    except Exception as e:
        print(f"Error loading table {table}: {e}")
        return
        
    end_time = time.time()
    print(f"DuckDB load complete in {end_time - start_time:.2f} seconds.")

def main():
    """Main data loading function."""
    print("--- Starting TPC-C Data Load ---")
    
    # 1. Generate Data
    dataframes = generate_mock_data()
    
    # 2. Load into PostgreSQL
    pg_conn = None
    try:
        pg_conn = create_pg_connection()
        create_tpcc_schema(pg_conn, is_postgres=True)
        load_data_postgres(pg_conn, dataframes)
    except Exception as e:
        print(f"PostgreSQL loading failed: {e}")
    finally:
        if pg_conn:
            pg_conn.close()
            print("PostgreSQL connection closed.")

    # 3. Load into DuckDB
    duck_conn = None
    try:
        duck_conn = connect_to_duckdb()
        create_tpcc_schema(duck_conn, is_postgres=False)
        load_data_duckdb(duck_conn, dataframes)
    except Exception as e:
        print(f"DuckDB loading failed: {e}")
    finally:
        if duck_conn:
            duck_conn.close()
            print("DuckDB connection closed.")
            
    print("\n--- TPC-C Data Load Complete ---")
    print(f"Data scale: {SCALE_FACTOR} warehouse(s)")
    print(f"DuckDB file: tpcc.duckdb")
    print(f"PostgreSQL schema: tpcc")

if __name__ == "__main__":
    main()

--- Starting TPC-C Data Load ---
Generating mock data for 2 warehouse(s)...
Mock data generated.
Successfully connected to PostgreSQL and ensured 'tpcc' schema exists.
Creating 9 TPC-C tables (PostgreSQL)...
Tables created.
Loading data into PostgreSQL...
  Loading warehouse (2 rows)...
  Loading district (20 rows)...
  Loading customer (60000 rows)...
Error loading table customer: value too long for type character(16)
CONTEXT:  COPY customer, line 3, column c_phone: "968.344.9512x5566"

PostgreSQL connection closed.
Successfully connected to DuckDB at 'tpcc.duckdb'.
Creating 9 TPC-C tables (DuckDB)...
Tables created.
Loading data into DuckDB...
  Loading warehouse (2 rows)...
  Loading district (20 rows)...
  Loading customer (60000 rows)...
  Skipping empty table: history
  Loading item (100000 rows)...
  Loading stock (200000 rows)...
  Skipping empty table: orders
  Skipping empty table: new_order
  Skipping empty table: order_line
DuckDB load complete in 1.80 seconds.
DuckDB conne

In [4]:
import psycopg2
import duckdb
import sys
import time
import random
from datetime import datetime

# --- Configuration ---
# Must match the scale factor used in load_tpcc_data.py
SCALE_FACTOR = 2 
DISTRICTS_PER_WAREHOUSE = 10
CUSTOMERS_PER_DISTRICT = 3000
ITEMS = 100000

# Number of transactions to run in the test
TOTAL_TRANSACTIONS = 200
# ---------------------

# SQL for PostgreSQL (uses %s)
PG_SQL = {
    "get_district": "SELECT d_next_o_id, d_tax FROM tpcc.district WHERE d_w_id = %s AND d_id = %s FOR UPDATE",
    "inc_next_o_id": "UPDATE tpcc.district SET d_next_o_id = d_next_o_id + 1 WHERE d_w_id = %s AND d_id = %s",
    "get_customer": "SELECT c_discount, c_last, c_credit FROM tpcc.customer WHERE c_w_id = %s AND c_d_id = %s AND c_id = %s",
    "create_order": "INSERT INTO tpcc.orders (o_id, o_d_id, o_w_id, o_c_id, o_entry_d, o_ol_cnt, o_all_local) VALUES (%s, %s, %s, %s, %s, %s, 1)",
    "create_new_order": "INSERT INTO tpcc.new_order (no_o_id, no_d_id, no_w_id) VALUES (%s, %s, %s)",
    "get_item": "SELECT i_price, i_name, i_data FROM tpcc.item WHERE i_id = %s",
    "get_stock": "SELECT s_quantity, s_data, s_dist_01 FROM tpcc.stock WHERE s_i_id = %s AND s_w_id = %s FOR UPDATE",
    "update_stock": "UPDATE tpcc.stock SET s_quantity = %s, s_ytd = s_ytd + %s, s_order_cnt = s_order_cnt + 1 WHERE s_i_id = %s AND s_w_id = %s",
    "create_order_line": "INSERT INTO tpcc.order_line (ol_o_id, ol_d_id, ol_w_id, ol_number, ol_i_id, ol_supply_w_id, ol_quantity, ol_amount, ol_dist_info) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)",

    "payment_w": "UPDATE tpcc.warehouse SET w_ytd = w_ytd + %s WHERE w_id = %s",
    "payment_d": "UPDATE tpcc.district SET d_ytd = d_ytd + %s WHERE d_w_id = %s AND d_id = %s",
    "payment_c": "UPDATE tpcc.customer SET c_balance = c_balance - %s, c_ytd_payment = c_ytd_payment + %s, c_payment_cnt = c_payment_cnt + 1 WHERE c_w_id = %s AND c_d_id = %s AND c_id = %s",
    "payment_h": "INSERT INTO tpcc.history (h_c_id, h_c_d_id, h_c_w_id, h_d_id, h_w_id, h_date, h_amount) VALUES (%s, %s, %s, %s, %s, %s, %s)",

    "order_status": "SELECT o_id, o_c_id, o_entry_d, o_carrier_id FROM tpcc.orders WHERE o_w_id = %s AND o_d_id = %s AND o_c_id = %s ORDER BY o_id DESC LIMIT 1",
    "order_status_lines": "SELECT ol_i_id, ol_supply_w_id, ol_quantity, ol_amount, ol_delivery_d FROM tpcc.order_line WHERE ol_w_id = %s AND ol_d_id = %s AND ol_o_id = %s"
}

# SQL for DuckDB (uses ?)
# NOTE: 'FOR UPDATE' is removed as DuckDB is single-writer and doesn't support it.
# This highlights a key difference in OLTP vs OLAP.
DUCK_SQL = {
    "get_district": "SELECT d_next_o_id, d_tax FROM district WHERE d_w_id = ? AND d_id = ?",
    "inc_next_o_id": "UPDATE district SET d_next_o_id = d_next_o_id + 1 WHERE d_w_id = ? AND d_id = ?",
    "get_customer": "SELECT c_discount, c_last, c_credit FROM customer WHERE c_w_id = ? AND c_d_id = ? AND c_id = ?",
    "create_order": "INSERT INTO orders (o_id, o_d_id, o_w_id, o_c_id, o_entry_d, o_ol_cnt, o_all_local) VALUES (?, ?, ?, ?, ?, ?, 1)",
    "create_new_order": "INSERT INTO new_order (no_o_id, no_d_id, no_w_id) VALUES (?, ?, ?)",
    "get_item": "SELECT i_price, i_name, i_data FROM item WHERE i_id = ?",
    "get_stock": "SELECT s_quantity, s_data, s_dist_01 FROM stock WHERE s_i_id = ? AND s_w_id = ?",
    "update_stock": "UPDATE stock SET s_quantity = ?, s_ytd = s_ytd + ?, s_order_cnt = s_order_cnt + 1 WHERE s_i_id = ? AND s_w_id = ?",
    "create_order_line": "INSERT INTO order_line (ol_o_id, ol_d_id, ol_w_id, ol_number, ol_i_id, ol_supply_w_id, ol_quantity, ol_amount, ol_dist_info) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)",

    "payment_w": "UPDATE warehouse SET w_ytd = w_ytd + ? WHERE w_id = ?",
    "payment_d": "UPDATE district SET d_ytd = d_ytd + ? WHERE d_w_id = ? AND d_id = ?",
    "payment_c": "UPDATE customer SET c_balance = c_balance - ?, c_ytd_payment = c_ytd_payment + ?, c_payment_cnt = c_payment_cnt + 1 WHERE c_w_id = ? AND c_d_id = ? AND c_id = ?",
    "payment_h": "INSERT INTO history (h_c_id, h_c_d_id, h_c_w_id, h_d_id, h_w_id, h_date, h_amount) VALUES (?, ?, ?, ?, ?, ?, ?)",

    "order_status": "SELECT o_id, o_c_id, o_entry_d, o_carrier_id FROM orders WHERE o_w_id = ? AND o_d_id = ? AND o_c_id = ? ORDER BY o_id DESC LIMIT 1",
    "order_status_lines": "SELECT ol_i_id, ol_supply_w_id, ol_quantity, ol_amount, ol_delivery_d FROM order_line WHERE ol_w_id = ? AND ol_d_id = ? AND ol_o_id = ?"
}

def create_pg_connection():
    """Creates a psycopg2 connection to PostgreSQL."""
    try:
        conn = psycopg2.connect(
            dbname='postgres',
            user='postgres',
            password='I4NAennDnRh1uO9z2y0k6pMfAf',
            host='localhost',
            port='5432'
        )
        conn.autocommit = False # We MUST control transactions
        return conn
    except Exception as e:
        print(f"Error: Could not connect to PostgreSQL: {e}")
        sys.exit(1)

def connect_to_duckdb():
    """Connects to the DuckDB database file (read-write)."""
    try:
        conn = duckdb.connect(database='tpcc.duckdb', read_only=False)
        return conn
    except Exception as e:
        print(f"Error: Could not connect to DuckDB: {e}")
        sys.exit(1)

# --- Transaction Functions ---

def run_new_order(conn, sql, is_postgres):
    """
    Simplified TPC-C New-Order Transaction
    """
    w_id = random.randint(1, SCALE_FACTOR)
    d_id = random.randint(1, DISTRICTS_PER_WAREHOUSE)
    c_id = random.randint(1, CUSTOMERS_PER_DISTRICT)
    ol_cnt = random.randint(5, 15)
    
    cursor = conn.cursor()
    
    try:
        # Get District
        cursor.execute(sql["get_district"], (w_id, d_id))
        d_next_o_id, d_tax = cursor.fetchone()
        
        # Increment District next_o_id
        cursor.execute(sql["inc_next_o_id"], (w_id, d_id))
        
        # Get Customer
        cursor.execute(sql["get_customer"], (w_id, d_id, c_id))
        c_discount, c_last, c_credit = cursor.fetchone()

        # Create Order
        o_id = d_next_o_id
        o_entry_d = datetime.now()
        cursor.execute(sql["create_order"], (o_id, d_id, w_id, c_id, o_entry_d, ol_cnt))
        
        # Create NewOrder
        cursor.execute(sql["create_new_order"], (o_id, d_id, w_id))
        
        total_amount = 0
        
        # Process Order Lines
        for ol_number in range(1, ol_cnt + 1):
            i_id = random.randint(1, ITEMS)
            
            cursor.execute(sql["get_item"], (i_id,))
            i_price, i_name, i_data = cursor.fetchone()
            
            cursor.execute(sql["get_stock"], (i_id, w_id))
            s_quantity, s_data, s_dist_01 = cursor.fetchone()
            
            if s_quantity > 10:
                s_quantity = s_quantity - 1
            else:
                s_quantity = s_quantity - 1 + 91 # Simulate restock
            
            cursor.execute(sql["update_stock"], (s_quantity, 1, i_id, w_id))
            
            ol_amount = 1 * i_price
            total_amount += ol_amount
            
            cursor.execute(sql["create_order_line"], (o_id, d_id, w_id, ol_number, i_id, w_id, 1, ol_amount, s_dist_01))
            
        if is_postgres:
            conn.commit()
        else:
            conn.commit() # DuckDB also needs commit
            
        return True
        
    except Exception as e:
        if is_postgres: conn.rollback()
        # DuckDB auto-rolls back on error in a transaction
        print(f"  [Error] New Order TX failed: {e}")
        return False
    finally:
        cursor.close()


def run_payment(conn, sql, is_postgres):
    """
    Simplified TPC-C Payment Transaction
    """
    w_id = random.randint(1, SCALE_FACTOR)
    d_id = random.randint(1, DISTRICTS_PER_WAREHOUSE)
    c_id = random.randint(1, CUSTOMERS_PER_DISTRICT)
    h_amount = round(random.uniform(1.00, 5000.00), 2)
    h_date = datetime.now()
    
    cursor = conn.cursor()
    
    try:
        cursor.execute(sql["payment_w"], (h_amount, w_id))
        cursor.execute(sql["payment_d"], (h_amount, w_id, d_id))
        cursor.execute(sql["payment_c"], (h_amount, h_amount, w_id, d_id, c_id))
        cursor.execute(sql["payment_h"], (c_id, d_id, w_id, d_id, w_id, h_date, h_amount))
        
        if is_postgres:
            conn.commit()
        else:
            conn.commit()
        return True

    except Exception as e:
        if is_postgres: conn.rollback()
        print(f"  [Error] Payment TX failed: {e}")
        return False
    finally:
        cursor.close()

def run_order_status(conn, sql):
    """
    Simplified TPC-C Order-Status Transaction (Read-Only)
    """
    w_id = random.randint(1, SCALE_FACTOR)
    d_id = random.randint(1, DISTRICTS_PER_WAREHOUSE)
    c_id = random.randint(1, CUSTOMERS_PER_DISTRICT)
    
    cursor = conn.cursor()
    
    try:
        # Find latest order for customer
        cursor.execute(sql["order_status"], (w_id, d_id, c_id))
        order = cursor.fetchone()
        
        if order:
            o_id, o_c_id, o_entry_d, o_carrier_id = order
            # Get order lines
            cursor.execute(sql["order_status_lines"], (w_id, d_id, o_id))
            order_lines = cursor.fetchall()
            return True # Found order
        else:
            return True # No order found is still a successful "read"

    except Exception as e:
        print(f"  [Error] Order Status TX failed: {e}")
        return False
    finally:
        cursor.close()

def run_benchmark(db_name, conn, sql_map, is_postgres):
    """Runs the mixed transaction workload against the given DB."""
    
    print(f"\n--- Running TPC-C Benchmark on {db_name} ---")
    
    tx_counts = {"new_order": 0, "payment": 0, "order_status": 0}
    tx_failures = {"new_order": 0, "payment": 0, "order_status": 0}
    
    start_time = time.time()
    
    for i in range(TOTAL_TRANSACTIONS):
        # TPC-C Transaction Mix: 45% NewOrder, 43% Payment, 4% OrderStatus, 4% Delivery, 4% StockLevel
        # Simplified Mix: 45% NewOrder, 45% Payment, 10% OrderStatus
        
        val = random.uniform(0, 100)
        
        if val <= 45:
            # --- New Order ---
            tx_counts["new_order"] += 1
            if not run_new_order(conn, sql_map, is_postgres):
                tx_failures["new_order"] += 1
                
        elif val <= 90:
            # --- Payment ---
            tx_counts["payment"] += 1
            if not run_payment(conn, sql_map, is_postgres):
                tx_failures["payment"] += 1
        
        else:
            # --- Order Status ---
            tx_counts["order_status"] += 1
            if not run_order_status(conn, sql_map):
                tx_failures["order_status"] += 1

    end_time = time.time()
    
    total_time = end_time - start_time
    tps = TOTAL_TRANSACTIONS / total_time
    
    print(f"--- {db_name} Results ---")
    print(f"  Total Transactions: {TOTAL_TRANSACTIONS}")
    print(f"  Total Time: {total_time:.2f} seconds")
    print(f"  Transactions Per Second (TPS): {tps:.2f}")
    print("  Transaction Mix (Success/Total):")
    print(f"    New Order:    {tx_counts['new_order'] - tx_failures['new_order']} / {tx_counts['new_order']}")
    print(f"    Payment:      {tx_counts['payment'] - tx_failures['payment']} / {tx_counts['payment']}")
    print(f"    Order Status: {tx_counts['order_status'] - tx_failures['order_status']} / {tx_counts['order_status']}")

def main():
    """Main execution function."""
    print("--- Starting TPC-C Simple Transaction Test ---")
    
    # 1. Run on PostgreSQL
    pg_conn = None
    try:
        pg_conn = create_pg_connection()
        # Set schema for this session
        with pg_conn.cursor() as cursor:
            cursor.execute("SET search_path TO tpcc, public;")
        run_benchmark("PostgreSQL", pg_conn, PG_SQL, is_postgres=True)
    except Exception as e:
        print(f"PostgreSQL benchmark failed: {e}")
    finally:
        if pg_conn:
            pg_conn.close()
            print("\nPostgreSQL connection closed.")

    # 2. Run on DuckDB
    duck_conn = None
    try:
        duck_conn = connect_to_duckdb()
        run_benchmark("DuckDB", duck_conn, DUCK_SQL, is_postgres=False)
    except Exception as e:
        print(f"DuckDB benchmark failed: {e}")
    finally:
        if duck_conn:
            duck_conn.close()
            print("\nDuckDB connection closed.")
            
    print("\n--- TPC-C Test Complete ---")

if __name__ == "__main__":
    main()


--- Starting TPC-C Simple Transaction Test ---

--- Running TPC-C Benchmark on PostgreSQL ---
  [Error] New Order TX failed: cannot unpack non-iterable NoneType object
  [Error] New Order TX failed: cannot unpack non-iterable NoneType object
  [Error] New Order TX failed: cannot unpack non-iterable NoneType object
  [Error] New Order TX failed: cannot unpack non-iterable NoneType object
  [Error] New Order TX failed: cannot unpack non-iterable NoneType object
  [Error] New Order TX failed: cannot unpack non-iterable NoneType object
  [Error] New Order TX failed: cannot unpack non-iterable NoneType object
  [Error] New Order TX failed: cannot unpack non-iterable NoneType object
  [Error] New Order TX failed: cannot unpack non-iterable NoneType object
  [Error] New Order TX failed: cannot unpack non-iterable NoneType object
  [Error] New Order TX failed: cannot unpack non-iterable NoneType object
  [Error] New Order TX failed: cannot unpack non-iterable NoneType object
  [Error] New Orde

Since we've already seen that OLTP use cases are much faster on PostgreSQL from our previous assignment, let us compare OLAP use cases instead.

We can use the industry standard TPC-H setup for that, which already comes with data prepared. You see the TPC-H schema here: https://docs.snowflake.com/en/_images/sample-data-tpch-schema.png

It ultimately simulates data from a "store" which receives many orders from multiple nations and customers, for various products or parts.  

In [11]:
%%capture
%pip install notebook duckdb==1.3 jupysql matplotlib duckdb-engine psycopg2;
%config SqlMagic.autopandas = True;
%config SqlMagic.feedback = False;
%config SqlMagic.displaycon = False;
import duckdb;
import pandas as pd
import zipfile;
import os;
from io import StringIO
import pandas as pd

# Stop truncating rows (show all)
pd.set_option('display.max_rows', None)

%load_ext sql
conn = duckdb.connect(database='tpch.duckdb')
%sql conn --alias duckdb
%sql duckdb:///:default:

Let's start with DuckDB, the embedded OLAP database, which should come out on top in this comparison. 

In [2]:
%%sql
INSTALL tpch;
LOAD tpch;

,Success


The above code will install TPC-H which actually comes built in with functions for generating data on DuckDB.

The TPC-H implementation on DuckDB will generate data for a standard scale of 1 (there are multiple scales used in TPC-H)

In [3]:
%%sql
CALL dbgen(sf = 0.1);

,Success


Now that we have the data for TPC-H, let's see how fast DuckDB is at running the benchmarks.

TPC-H doesn't just come with data generating functions. It also comes with a set of 22 queries which are standard for benchmarking. We can see the queries by running the commands for the extension below:

In [17]:
%%sql
SELECT query FROM tpch_queries();

,query
0,"SELECT\n l_returnflag,\n l_linestatus,\n..."
1,"SELECT\n s_acctbal,\n s_name,\n n_nam..."
2,"SELECT\n l_orderkey,\n sum(l_extendedpri..."
3,"SELECT\n o_orderpriority,\n count(*) AS ..."
4,"SELECT\n n_name,\n sum(l_extendedprice *..."
5,SELECT\n sum(l_extendedprice * l_discount) ...
6,"SELECT\n supp_nation,\n cust_nation,\n ..."
7,"SELECT\n o_year,\n sum(\n CASE WH..."
8,"SELECT\n nation,\n o_year,\n sum(amou..."
9,"SELECT\n c_custkey,\n c_name,\n sum(l..."


You can see the queries are largely selects, with several joins. Let's give them a run!

In [50]:
%%timeit
%%sql
PRAGMA tpch(1);
PRAGMA tpch(2);
PRAGMA tpch(3);
PRAGMA tpch(4);
PRAGMA tpch(5);
PRAGMA tpch(6);
PRAGMA tpch(7);
PRAGMA tpch(8);
PRAGMA tpch(9);
PRAGMA tpch(10);
PRAGMA tpch(11);
PRAGMA tpch(12);
PRAGMA tpch(13);
PRAGMA tpch(14);
PRAGMA tpch(15);
PRAGMA tpch(16);
PRAGMA tpch(17);
PRAGMA tpch(18);
PRAGMA tpch(19);
PRAGMA tpch(20);
PRAGMA tpch(21);

1.14 s ± 17.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


We now have an estimate for how long typical analytics / aggregation / read queries take on an OLAP embedded database. Now let's see how Postgres handles this.

Note - this isn't an apples to apples comparison, or a fair benchmark. It's largely just to illustrate the strengths of OLAP vs. OLTP use cases. There are many variables we haven't controlled here, such as network delay (DuckDB is embedded, Postgres is not), machine/server memory (DuckDB runs locally, Postgres runs on a server) and so on. But, you'll still see a difference! Let's start by creating the tables in PostgreSQL.

In [31]:
import psycopg2
from psycopg2 import Error

def create_pg_connection():
    """Creates a psycopg2 connection to PostgreSQL."""
    try:
        conn = psycopg2.connect(
            dbname='postgres',
            user='postgres',
            password='I4NAennDnRh1uO9z2y0k6pMfAf',
            host='localhost',
            port='5432'
        )
        return conn
    except Exception as e:
        print(f"Error: Could not connect to PostgreSQL using psycopg2.")
        print(f"Details: {e}")
        print("Please check the hardcoded connection details in the create_pg_connection function.")
        sys.exit(1)


def execute_query(conn, sql_query, fetch = False):
    cursor = cursor = conn.cursor()
    cursor.execute(sql_query)
    if fetch == True:
        results = cursor.fetchall()
        print(results)
        return results
    conn.commit()
    cursor.close()
    conn.close()

Next, let's insert data that we have into PostgreSQL.

In [74]:
DUCKDB_PATH = 'tpch.duckdb'

def connect_to_duckdb():
    """Connects to the DuckDB database."""
    try:
        # Connect to the DuckDB file
        conn = duckdb.connect(database=DUCKDB_PATH, read_only=True)
        print(f"Successfully connected to DuckDB at '{DUCKDB_PATH}'.")
        return conn
    except Exception as e:
        print(f"Error: Could not connect to DuckDB at '{DUCKDB_PATH}'.")
        print(f"Details: {e}")
        sys.exit(1)

In [79]:
import duckdb
import pandas as pd
import psycopg2  # Use psycopg2 directly
import sys
import time
import io  # Needed for in-memory buffer for COPY

# --- Configuration ---
DUCKDB_PATH = 'tpch.duckdb'

# TPC-H tables in an order that respects foreign key constraints
# (e.g., region and nation must exist before customer and supplier)
TPCH_TABLES = [
    'region',
    'nation',
    'part',
    'supplier',
    'partsupp',
    'customer',
    'orders',
    'lineitem'
]
# ---------------------

def connect_to_duckdb():
    """Connects to the DuckDB database."""
    try:
        # Connect to the DuckDB file
        conn = duckdb.connect(database=DUCKDB_PATH, read_only=True)
        print(f"Successfully connected to DuckDB at '{DUCKDB_PATH}'.")
        return conn
    except Exception as e:
        print(f"Error: Could not connect to DuckDB at '{DUCKDB_PATH}'.")
        print(f"Details: {e}")
        sys.exit(1)

def map_duckdb_to_postgres_type(duckdb_type):
    """Maps DuckDB data types to compatible PostgreSQL data types."""
    duckdb_type = duckdb_type.upper()
    
    if duckdb_type == 'VARCHAR':
        return 'TEXT'
    if duckdb_type == 'DOUBLE':
        return 'DOUBLE PRECISION'
    if duckdb_type.startswith('DECIMAL') or duckdb_type.startswith('NUMERIC'):
        return duckdb_type  # e.g., DECIMAL(15, 2)
    if duckdb_type in ('INTEGER', 'BIGINT', 'DATE', 'TIMESTAMP'):
        return duckdb_type

    # Default fallback
    return 'TEXT'

def transfer_table(table_name, duck_conn, pg_conn):
    """
    Transfers a single table from DuckDB to PostgreSQL (into 'tpch' schema) using psycopg2.COPY.
    - Extracts data from DuckDB into a pandas DataFrame.
    - Infers schema from DuckDB.
    - Re-creates the table in PostgreSQL's 'tpch' schema.
    - Loads the data using COPY for high performance.
    """
    print(f"\n--- Processing table: {table_name} ---")
    
    try:
        # 1. Extract from DuckDB
        print(f"  Extracting '{table_name}' from DuckDB...")
        start_time = time.time()
        query = f"SELECT * FROM {table_name}"
        # Use .df() to efficiently load the query result into a pandas DataFrame
        df = duck_conn.query(query).df()
        extract_time = time.time() - start_time
        print(f"  Extracted {len(df)} rows in {extract_time:.2f} seconds.")

        if df.empty:
            print(f"  Skipping load for '{table_name}' as it contains no data.")
            return

        # 2. Get schema from DuckDB and create DDL
        print(f"  Generating schema for '{table_name}'...")
        schema_df = duck_conn.query(f"DESCRIBE SELECT * FROM {table_name}").df()
        columns_sql = []
        for _, row in schema_df.iterrows():
            col_name = row['column_name']
            pg_type = map_duckdb_to_postgres_type(row['column_type'])
            # Quote column names to handle special characters or keywords
            columns_sql.append(f'"{col_name}" {pg_type}')
        
        # Create table in the 'tpch' schema
        create_table_sql = f"CREATE TABLE tpch.{table_name} ({', '.join(columns_sql)});"


        # 3. Load into PostgreSQL using psycopg2.copy_expert
        print(f"  Loading '{table_name}' into PostgreSQL schema 'tpch' using COPY...")
        start_time = time.time()
        
        buffer = io.StringIO()
        # Write dataframe to buffer as CSV with tab delimiter
        # Use na_rep='\\N' for NULL values, which COPY understands
        df.to_csv(buffer, index=False, header=False, sep='\t', na_rep='\\N')
        buffer.seek(0)
        
        with pg_conn.cursor() as cursor:
            # Drop table from the 'tpch' schema
            cursor.execute(f"DROP TABLE IF EXISTS tpch.{table_name} CASCADE;")
            
            # Create the new table (already schema-qualified)
            cursor.execute(create_table_sql)
            
            # Use copy_expert for STDIN copy into the 'tpch' schema
            copy_sql = f"COPY tpch.{table_name} FROM STDIN WITH (FORMAT CSV, DELIMITER E'\\t', NULL '\\N')"
            cursor.copy_expert(copy_sql, buffer)
            
        pg_conn.commit()
        
        load_time = time.time() - start_time
        print(f"  Successfully loaded '{table_name}' into 'tpch' in {load_time:.2f} seconds.")

    except Exception as e:
        print(f"Error processing table '{table_name}': {e}")
        # Rollback any partial transaction on error
        try:
            pg_conn.rollback()
        except Exception as rb_e:
            print(f"  Error during rollback: {rb_e}")
        print("Stopping script.")
        sys.exit(1)

def main():
    """Main execution function."""
    print("Starting TPC-H data transfer from DuckDB to PostgreSQL (schema 'tpch')...")
    
    pg_conn = create_pg_connection()  # Changed to psycopg2 connection
    duck_conn = connect_to_duckdb()
    
    script_start_time = time.time()
    
    try:
        with duck_conn:
            for table in TPCH_TABLES:
                transfer_table(table, duck_conn, pg_conn)  # Pass psycopg2 conn
    finally:
        # Ensure connections are closed
        if pg_conn:
            pg_conn.close()
            print("\nPostgreSQL connection closed.")
            
    script_end_time = time.time()
    total_time = script_end_time - script_start_time
    
    print(f"\n--- Transfer Complete ---")
    print(f"All {len(TPCH_TABLES)} tables transferred successfully to schema 'tpch'.")
    print(f"Total time: {total_time:.2f} seconds.")

if __name__ == "__main__":
    main()



Starting TPC-H data transfer from DuckDB to PostgreSQL (schema 'tpch')...
Successfully connected to PostgreSQL using psycopg2.
Successfully connected to DuckDB at 'tpch.duckdb'.

--- Processing table: region ---
  Extracting 'region' from DuckDB...
  Extracted 5 rows in 0.00 seconds.
  Generating schema for 'region'...
  Loading 'region' into PostgreSQL schema 'tpch' using COPY...
  Successfully loaded 'region' into 'tpch' in 0.02 seconds.

--- Processing table: nation ---
  Extracting 'nation' from DuckDB...
  Extracted 25 rows in 0.00 seconds.
  Generating schema for 'nation'...
  Loading 'nation' into PostgreSQL schema 'tpch' using COPY...
  Successfully loaded 'nation' into 'tpch' in 0.01 seconds.

--- Processing table: part ---
  Extracting 'part' from DuckDB...
  Extracted 200000 rows in 0.15 seconds.
  Generating schema for 'part'...
  Loading 'part' into PostgreSQL schema 'tpch' using COPY...
  Successfully loaded 'part' into 'tpch' in 1.41 seconds.

--- Processing table: suppl

Finally, let's run the queries in PostgreSQL!

In [37]:
querystring = """
SELECT
    l_returnflag,
    l_linestatus,
    sum(l_quantity) AS sum_qty,
    sum(l_extendedprice) AS sum_base_price,
    sum(l_extendedprice * (1 - l_discount)) AS sum_disc_price,
    sum(l_extendedprice * (1 - l_discount) * (1 + l_tax)) AS sum_charge,
    avg(l_quantity) AS avg_qty,
    avg(l_extendedprice) AS avg_price,
    avg(l_discount) AS avg_disc,
    count(*) AS count_order
FROM
    tpch.lineitem
WHERE
    l_shipdate <= CAST('1998-09-02' AS date)
GROUP BY
    l_returnflag,
    l_linestatus
ORDER BY
    l_returnflag,
    l_linestatus;

SELECT
    s_acctbal,
    s_name,
    n_name,
    p_partkey,
    p_mfgr,
    s_address,
    s_phone,
    s_comment
FROM
    tpch.part,
    tpch.supplier,
    tpch.partsupp,
    tpch.nation,
    tpch.region
WHERE
    p_partkey = ps_partkey
    AND s_suppkey = ps_suppkey
    AND p_size = 15
    AND p_type LIKE '%BRASS'
    AND s_nationkey = n_nationkey
    AND n_regionkey = r_regionkey
    AND r_name = 'EUROPE'
    AND ps_supplycost = (
        SELECT
            min(ps_supplycost)
        FROM
            tpch.partsupp,
            tpch.supplier,
            tpch.nation,
            tpch.region
        WHERE
            p_partkey = ps_partkey
            AND s_suppkey = ps_suppkey
            AND s_nationkey = n_nationkey
            AND n_regionkey = r_regionkey
            AND r_name = 'EUROPE')
ORDER BY
    s_acctbal DESC,
    n_name,
    s_name,
    p_partkey
LIMIT 100;

SELECT
    l_orderkey,
    sum(l_extendedprice * (1 - l_discount)) AS revenue,
    o_orderdate,
    o_shippriority
FROM
    tpch.customer,
    tpch.orders,
    tpch.lineitem
WHERE
    c_mktsegment = 'BUILDING'
    AND c_custkey = o_custkey
    AND l_orderkey = o_orderkey
    AND o_orderdate < CAST('1995-03-15' AS date)
    AND l_shipdate > CAST('1995-03-15' AS date)
GROUP BY
    l_orderkey,
    o_orderdate,
    o_shippriority
ORDER BY
    revenue DESC,
    o_orderdate
LIMIT 10;

SELECT
    o_orderpriority,
    count(*) AS order_count
FROM
    tpch.orders
WHERE
    o_orderdate >= CAST('1993-07-01' AS date)
    AND o_orderdate < CAST('1993-10-01' AS date)
    AND EXISTS (
        SELECT
            *
        FROM
            tpch.lineitem
        WHERE
            l_orderkey = o_orderkey
            AND l_commitdate < l_receiptdate)
GROUP BY
    o_orderpriority
ORDER BY
    o_orderpriority;

SELECT
    n_name,
    sum(l_extendedprice * (1 - l_discount)) AS revenue
FROM
    tpch.customer,
    tpch.orders,
    tpch.lineitem,
    tpch.supplier,
    tpch.nation,
    tpch.region
WHERE
    c_custkey = o_custkey
    AND l_orderkey = o_orderkey
    AND l_suppkey = s_suppkey
    AND c_nationkey = s_nationkey
    AND s_nationkey = n_nationkey
    AND n_regionkey = r_regionkey
    AND r_name = 'ASIA'
    AND o_orderdate >= CAST('1994-01-01' AS date)
    AND o_orderdate < CAST('1995-01-01' AS date)
GROUP BY
    n_name
ORDER BY
    revenue DESC;

SELECT
    sum(l_extendedprice * l_discount) AS revenue
FROM
    tpch.lineitem
WHERE
    l_shipdate >= CAST('1994-01-01' AS date)
    AND l_shipdate < CAST('1995-01-01' AS date)
    AND l_discount BETWEEN 0.05
    AND 0.07
    AND l_quantity < 24;

SELECT
    supp_nation,
    cust_nation,
    l_year,
    sum(volume) AS revenue
FROM (
    SELECT
        n1.n_name AS supp_nation,
        n2.n_name AS cust_nation,
        extract(year FROM l_shipdate) AS l_year,
        l_extendedprice * (1 - l_discount) AS volume
    FROM
        tpch.supplier,
        tpch.lineitem,
        tpch.orders,
        tpch.customer,
        tpch.nation n1,
        tpch.nation n2
    WHERE
        s_suppkey = l_suppkey
        AND o_orderkey = l_orderkey
        AND c_custkey = o_custkey
        AND s_nationkey = n1.n_nationkey
        AND c_nationkey = n2.n_nationkey
        AND ((n1.n_name = 'FRANCE'
                AND n2.n_name = 'GERMANY')
            OR (n1.n_name = 'GERMANY'
                AND n2.n_name = 'FRANCE'))
        AND l_shipdate BETWEEN CAST('1995-01-01' AS date)
        AND CAST('1996-12-31' AS date)) AS shipping
GROUP BY
    supp_nation,
    cust_nation,
    l_year
ORDER BY
    supp_nation,
    cust_nation,
    l_year;

SELECT
    o_year,
    sum(
        CASE WHEN nation = 'BRAZIL' THEN
            volume
        ELSE
            0
        END) / sum(volume) AS mkt_share
FROM (
    SELECT
        extract(year FROM o_orderdate) AS o_year,
        l_extendedprice * (1 - l_discount) AS volume,
        n2.n_name AS nation
    FROM
        tpch.part,
        tpch.supplier,
        tpch.lineitem,
        tpch.orders,
        tpch.customer,
        tpch.nation n1,
        tpch.nation n2,
        tpch.region
    WHERE
        p_partkey = l_partkey
        AND s_suppkey = l_suppkey
        AND l_orderkey = o_orderkey
        AND o_custkey = c_custkey
        AND c_nationkey = n1.n_nationkey
        AND n1.n_regionkey = r_regionkey
        AND r_name = 'AMERICA'
        AND s_nationkey = n2.n_nationkey
        AND o_orderdate BETWEEN CAST('1995-01-01' AS date)
        AND CAST('1996-12-31' AS date)
        AND p_type = 'ECONOMY ANODIZED STEEL') AS all_nations
GROUP BY
    o_year
ORDER BY
    o_year;

SELECT
    nation,
    o_year,
    sum(amount) AS sum_profit
FROM (
    SELECT
        n_name AS nation,
        extract(year FROM o_orderdate) AS o_year,
        l_extendedprice * (1 - l_discount) - ps_supplycost * l_quantity AS amount
    FROM
        tpch.part,
        tpch.supplier,
        tpch.lineitem,
        tpch.partsupp,
        tpch.orders,
        tpch.nation
    WHERE
        s_suppkey = l_suppkey
        AND ps_suppkey = l_suppkey
        AND ps_partkey = l_partkey
        AND p_partkey = l_partkey
        AND o_orderkey = l_orderkey
        AND s_nationkey = n_nationkey
        AND p_name LIKE '%green%') AS profit
GROUP BY
    nation,
    o_year
ORDER BY
    nation,
    o_year DESC;

SELECT
    c_custkey,
    c_name,
    sum(l_extendedprice * (1 - l_discount)) AS revenue,
    c_acctbal,
    n_name,
    c_address,
    c_phone,
    c_comment
FROM
    tpch.customer,
    tpch.orders,
    tpch.lineitem,
    tpch.nation
WHERE
    c_custkey = o_custkey
    AND l_orderkey = o_orderkey
    AND o_orderdate >= CAST('1993-10-01' AS date)
    AND o_orderdate < CAST('1994-01-01' AS date)
    AND l_returnflag = 'R'
    AND c_nationkey = n_nationkey
GROUP BY
    c_custkey,
    c_name,
    c_acctbal,
    c_phone,
    n_name,
    c_address,
    c_comment
ORDER BY
    revenue DESC
LIMIT 20;

SELECT
    ps_partkey,
    sum(ps_supplycost * ps_availqty) AS value
FROM
    tpch.partsupp,
    tpch.supplier,
    tpch.nation
WHERE
    ps_suppkey = s_suppkey
    AND s_nationkey = n_nationkey
    AND n_name = 'GERMANY'
GROUP BY
    ps_partkey
HAVING
    sum(ps_supplycost * ps_availqty) > (
        SELECT
            sum(ps_supplycost * ps_availqty) * 0.0001000000
        FROM
            tpch.partsupp,
            tpch.supplier,
            tpch.nation
        WHERE
            ps_suppkey = s_suppkey
            AND s_nationkey = n_nationkey
            AND n_name = 'GERMANY')
ORDER BY
    value DESC;

SELECT
    l_shipmode,
    sum(
        CASE WHEN o_orderpriority = '1-URGENT'
            OR o_orderpriority = '2-HIGH' THEN
            1
        ELSE
            0
        END) AS high_line_count,
    sum(
        CASE WHEN o_orderpriority <> '1-URGENT'
            AND o_orderpriority <> '2-HIGH' THEN
            1
        ELSE
            0
        END) AS low_line_count
FROM
    tpch.orders,
    tpch.lineitem
WHERE
    o_orderkey = l_orderkey
    AND l_shipmode IN ('MAIL', 'SHIP')
    AND l_commitdate < l_receiptdate
    AND l_shipdate < l_commitdate
    AND l_receiptdate >= CAST('1994-01-01' AS date)
    AND l_receiptdate < CAST('1995-01-01' AS date)
GROUP BY
    l_shipmode
ORDER BY
    l_shipmode;

SELECT
    c_count,
    count(*) AS custdist
FROM (
    SELECT
        c_custkey,
        count(o_orderkey)
    FROM
        tpch.customer
    LEFT OUTER JOIN tpch.orders ON c_custkey = o_custkey
    AND o_comment NOT LIKE '%special%requests%'
GROUP BY
    c_custkey) AS c_orders (c_custkey,
        c_count)
GROUP BY
    c_count
ORDER BY
    custdist DESC,
    c_count DESC;

SELECT
    100.00 * sum(
        CASE WHEN p_type LIKE 'PROMO%' THEN
            l_extendedprice * (1 - l_discount)
        ELSE
            0
        END) / sum(l_extendedprice * (1 - l_discount)) AS promo_revenue
FROM
    tpch.lineitem,
    tpch.part
WHERE
    l_partkey = p_partkey
    AND l_shipdate >= date '1995-09-01'
    AND l_shipdate < CAST('1995-10-01' AS date);

WITH revenue AS (
    SELECT
        l_suppkey AS supplier_no,
        sum(l_extendedprice * (1 - l_discount)) AS total_revenue
    FROM
        tpch.lineitem
    WHERE
        l_shipdate >= CAST('1996-01-01' AS date)
        AND l_shipdate < CAST('1996-04-01' AS date)
    GROUP BY
        supplier_no
)
SELECT
    s_suppkey,
    s_name,
    s_address,
    s_phone,
    total_revenue
FROM
    tpch.supplier,
    revenue
WHERE
    s_suppkey = supplier_no
    AND total_revenue = (
        SELECT
            max(total_revenue)
        FROM revenue)
ORDER BY
    s_suppkey;

SELECT
    p_brand,
    p_type,
    p_size,
    count(DISTINCT ps_suppkey) AS supplier_cnt
FROM
    tpch.partsupp,
    tpch.part
WHERE
    p_partkey = ps_partkey
    AND p_brand <> 'Brand#45'
    AND p_type NOT LIKE 'MEDIUM POLISHED%'
    AND p_size IN (49, 14, 23, 45, 19, 3, 36, 9)
    AND ps_suppkey NOT IN (
        SELECT
            s_suppkey
        FROM
            tpch.supplier
        WHERE
            s_comment LIKE '%Customer%Complaints%')
GROUP BY
    p_brand,
    p_type,
    p_size
ORDER BY
    supplier_cnt DESC,
    p_brand,
    p_type,
    p_size;

SELECT
    sum(l_extendedprice) / 7.0 AS avg_yearly
FROM
    tpch.lineitem,
    tpch.part
WHERE
    p_partkey = l_partkey
    AND p_brand = 'Brand#23'
    AND p_container = 'MED BOX'
    AND l_quantity < (
        SELECT
            0.2 * avg(l_quantity)
        FROM
            tpch.lineitem
        WHERE
            l_partkey = p_partkey);

SELECT
    c_name,
    c_custkey,
    o_orderkey,
    o_orderdate,
    o_totalprice,
    sum(l_quantity)
FROM
    tpch.customer,
    tpch.orders,
    tpch.lineitem
WHERE
    o_orderkey IN (
        SELECT
            l_orderkey
        FROM
            tpch.lineitem
        GROUP BY
            l_orderkey
        HAVING
            sum(l_quantity) > 300)
    AND c_custkey = o_custkey
    AND o_orderkey = l_orderkey
GROUP BY
    c_name,
    c_custkey,
    o_orderkey,
    o_orderdate,
    o_totalprice
ORDER BY
    o_totalprice DESC,
    o_orderdate
LIMIT 100;

SELECT
    sum(l_extendedprice * (1 - l_discount)) AS revenue
FROM
    tpch.lineitem,
    tpch.part
WHERE (p_partkey = l_partkey
    AND p_brand = 'Brand#12'
    AND p_container IN ('SM CASE', 'SM BOX', 'SM PACK', 'SM PKG')
    AND l_quantity >= 1
    AND l_quantity <= 1 + 10
    AND p_size BETWEEN 1 AND 5
    AND l_shipmode IN ('AIR', 'AIR REG')
    AND l_shipinstruct = 'DELIVER IN PERSON')
    OR (p_partkey = l_partkey
        AND p_brand = 'Brand#23'
        AND p_container IN ('MED BAG', 'MED BOX', 'MED PKG', 'MED PACK')
        AND l_quantity >= 10
        AND l_quantity <= 10 + 10
        AND p_size BETWEEN 1 AND 10
        AND l_shipmode IN ('AIR', 'AIR REG')
        AND l_shipinstruct = 'DELIVER IN PERSON')
    OR (p_partkey = l_partkey
        AND p_brand = 'Brand#34'
        AND p_container IN ('LG CASE', 'LG BOX', 'LG PACK', 'LG PKG')
        AND l_quantity >= 20
        AND l_quantity <= 20 + 10
        AND p_size BETWEEN 1 AND 15
        AND l_shipmode IN ('AIR', 'AIR REG')
        AND l_shipinstruct = 'DELIVER IN PERSON');

SELECT
    s_name,
    s_address
FROM
    tpch.supplier,
    tpch.nation
WHERE
    s_suppkey IN (
        SELECT
            ps_suppkey
        FROM
            tpch.partsupp
        WHERE
            ps_partkey IN (
                SELECT
                    p_partkey
                FROM
                    tpch.part
                WHERE
                    p_name LIKE 'forest%')
                AND ps_availqty > (
                    SELECT
                        0.5 * sum(l_quantity)
                    FROM
                        tpch.lineitem
                    WHERE
                        l_partkey = ps_partkey
                        AND l_suppkey = ps_suppkey
                        AND l_shipdate >= CAST('1994-01-01' AS date)
                        AND l_shipdate < CAST('1995-01-01' AS date)))
            AND s_nationkey = n_nationkey
            AND n_name = 'CANADA'
    ORDER BY
        s_name;

SELECT
    s_name,
    count(*) AS numwait
FROM
    tpch.supplier,
    tpch.lineitem l1,
    tpch.orders,
    tpch.nation
WHERE
    s_suppkey = l1.l_suppkey
    AND o_orderkey = l1.l_orderkey
    AND o_orderstatus = 'F'
    AND l1.l_receiptdate > l1.l_commitdate
    AND EXISTS (
        SELECT
            *
        FROM
            tpch.lineitem l2
        WHERE
            l2.l_orderkey = l1.l_orderkey
            AND l2.l_suppkey <> l1.l_suppkey)
    AND NOT EXISTS (
        SELECT
            *
        FROM
            tpch.lineitem l3
        WHERE
            l3.l_orderkey = l1.l_orderkey
            AND l3.l_suppkey <> l1.l_suppkey
            AND l3.l_receiptdate > l3.l_commitdate)
    AND s_nationkey = n_nationkey
    AND n_name = 'SAUDI ARABIA'
GROUP BY
    s_name
ORDER BY
    numwait DESC,
    s_name
LIMIT 100;

SELECT
    cntrycode,
    count(*) AS numcust,
    sum(c_acctbal) AS totacctbal
FROM (
    SELECT
        substring(c_phone FROM 1 FOR 2) AS cntrycode,
        c_acctbal
    FROM
        tpch.customer
    WHERE
        substring(c_phone FROM 1 FOR 2) IN ('13', '31', '23', '29', '30', '18', '17')
        AND c_acctbal > (
            SELECT
                avg(c_acctbal)
            FROM
                tpch.customer
            WHERE
                c_acctbal > 0.00
                AND substring(c_phone FROM 1 FOR 2) IN ('13', '31', '23', '29', '30', '18', '17'))
            AND NOT EXISTS (
                SELECT
                    *
                FROM
                    tpch.orders
                WHERE
                    o_custkey = c_custkey)) AS custsale
GROUP BY
    cntrycode
ORDER BY
    cntrycode;
"""

In [ ]:
%%timeit -n 3
execute_query(conn=create_pg_connection(), sql_query = querystring, fetch = True)

You can see that DuckDB is much faster than Postgres on the same type of query for analytical reads.